# Checkpoint 2 (Week 2): Vision (and optional multimodal) Pipelines

This notebook implements exactly **two** Hugging Face `pipeline` vision/multimodal tasks:
1. **Object detection** (`object-detection`) — with bounding box visualization
2. **Image captioning / image-to-text** (`image-to-text`) — with caption visualization

It processes a **small image set (≥ 20 images)** from a Hugging Face dataset, visualizes outputs, and compares **two object detection models** with latency + qualitative differences (and a lightweight metric based on IoU vs ground truth).

In [ ]:
!pip install -U transformers datasets evaluate accelerate torch torchvision pillow matplotlib

In [ ]:
import time
import numpy as np
import pandas as pd

import torch
from transformers import pipeline
from datasets import load_dataset

from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)

DEVICE = 0 if torch.cuda.is_available() else -1
print('DEVICE:', 'cuda' if DEVICE == 0 else 'cpu')

In [ ]:
def avg_latency(fn, inputs, batch_size=4, repeats=1):
    """Average latency (seconds) per example (batched)."""
    n = len(inputs)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        _ = fn(inputs, batch_size=batch_size)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return float(np.mean(times) / n)


def iou_xyxy(a, b):
    """IoU for boxes in [xmin,ymin,xmax,ymax]."""
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    inter_w, inter_h = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0


def draw_detections(img, detections, score_thr=0.5, title=None):
    """Draw detection boxes on a PIL image."""
    im = img.convert('RGB').copy()
    draw = ImageDraw.Draw(im)

    # Try default font; if not available, PIL will still draw with a basic font.
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    for det in detections:
        score = float(det.get('score', 0.0))
        if score < score_thr:
            continue
        box = det['box']
        x1, y1, x2, y2 = box['xmin'], box['ymin'], box['xmax'], box['ymax']
        label = det.get('label', 'obj')

        draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
        txt = f"{label} {score:.2f}"
        # Background for text
        tw, th = draw.textbbox((0, 0), txt, font=font)[2:]
        draw.rectangle([x1, max(0, y1 - th - 4), x1 + tw + 6, y1], fill='red')
        draw.text((x1 + 3, max(0, y1 - th - 2)), txt, fill='white', font=font)

    if title:
        plt.figure(figsize=(10, 6))
        plt.imshow(im)
        plt.axis('off')
        plt.title(title)
        plt.show()
    return im

---
## Part A: Choose two vision tasks

Chosen tasks:
- **Object detection** (`object-detection`)
- **Image captioning** (`image-to-text`)

We use a Hugging Face dataset to create a small image set (≥ 20 images).

In [ ]:
# Load a labeled detection dataset with images (≥ 20)
# CPPE-5 contains PPE images with bounding boxes (helmet, vest, etc.)
# We'll use a small subset for fast prototyping.

ds = load_dataset('cppe-5')
print(ds)

split = 'test' if 'test' in ds else 'train'
N_IMAGES = 20
subset = ds[split].select(range(N_IMAGES))

# Each item has: image, objects (bboxes + category)
print('Using split:', split)
print('Subset size:', len(subset))

# Collect PIL images
images = [ex['image'] for ex in subset]

# Peek at one example structure
example = subset[0]
example.keys(), example['objects'].keys()

---
## Task 1: Object detection (`object-detection`)

We compare **two different detection models**:
- `facebook/detr-resnet-50` (DETR)
- `hustvl/yolos-tiny` (YOLOS Tiny)

Both return a list of detections with `box` (xmin, ymin, xmax, ymax), `label`, and `score`.

### Visualization requirement
We will draw bounding boxes and labels on at least 5 images.

In [ ]:
DET_MODEL_A = 'facebook/detr-resnet-50'
DET_MODEL_B = 'hustvl/yolos-tiny'

# Pipelines
det_a = pipeline('object-detection', model=DET_MODEL_A, device=DEVICE)
det_b = pipeline('object-detection', model=DET_MODEL_B, device=DEVICE)

# Wrapper to standardize call signature
# Some object-detection pipelines cannot batch variable-size images cleanly.
# We try batching first; if it fails, we fall back to per-image inference.

def det_predict(det_pipe, imgs, batch_size=4):
    try:
        return det_pipe(imgs, batch_size=batch_size)
    except RuntimeError as e:
        print('Batching failed; falling back to batch_size=1. Error was:')
        print(e)
        return [det_pipe(img, batch_size=1) for img in imgs]

# Run predictions for all 20 images
BATCH = 4
preds_a = det_predict(det_a, images, batch_size=BATCH)
preds_b = det_predict(det_b, images, batch_size=BATCH)

print('Predictions collected:', len(preds_a), len(preds_b))

In [ ]:
# Visualize detections on at least 5 images
SCORE_THR = 0.6
VIS_N = 5

for i in range(VIS_N):
    img = images[i]
    _ = draw_detections(img, preds_a[i], score_thr=SCORE_THR, title=f"DETR: {DET_MODEL_A} | image {i}")
    _ = draw_detections(img, preds_b[i], score_thr=SCORE_THR, title=f"YOLOS: {DET_MODEL_B} | image {i}")

In [ ]:
# Latency comparison (avg ms per image) on the same 20-image set
lat_a = avg_latency(lambda imgs, batch_size: det_predict(det_a, imgs, batch_size=batch_size), images, batch_size=BATCH, repeats=1)
lat_b = avg_latency(lambda imgs, batch_size: det_predict(det_b, imgs, batch_size=batch_size), images, batch_size=BATCH, repeats=1)

lat_table = pd.DataFrame([
    {'model': DET_MODEL_A, 'avg_latency_ms_per_image': lat_a * 1000.0},
    {'model': DET_MODEL_B, 'avg_latency_ms_per_image': lat_b * 1000.0},
]).sort_values('avg_latency_ms_per_image')

lat_table

In [ ]:
# Lightweight metric (since dataset has labels): best-IoU per ground-truth box
# CPPE-5 ground truth boxes are in COCO format [x, y, w, h]. We'll convert to xyxy.

def coco_to_xyxy(b):
    x, y, w, h = b
    return [x, y, x + w, y + h]


def best_iou_for_image(gt_boxes_xyxy, pred_boxes_xyxy):
    if len(gt_boxes_xyxy) == 0 or len(pred_boxes_xyxy) == 0:
        return 0.0
    # For each gt box, compute best IoU among predictions
    bests = []
    for g in gt_boxes_xyxy:
        bests.append(max(iou_xyxy(g, p) for p in pred_boxes_xyxy))
    return float(np.mean(bests))


def preds_to_xyxy(pred_list, score_thr=0.3):
    out = []
    for d in pred_list:
        if float(d.get('score', 0.0)) < score_thr:
            continue
        b = d['box']
        out.append([b['xmin'], b['ymin'], b['xmax'], b['ymax']])
    return out

ious_a, ious_b = [], []
for i in range(N_IMAGES):
    gt_boxes = [coco_to_xyxy(b) for b in subset[i]['objects']['bbox']]
    pa = preds_to_xyxy(preds_a[i], score_thr=0.3)
    pb = preds_to_xyxy(preds_b[i], score_thr=0.3)
    ious_a.append(best_iou_for_image(gt_boxes, pa))
    ious_b.append(best_iou_for_image(gt_boxes, pb))

metric_table = pd.DataFrame([
    {'model': DET_MODEL_A, 'mean_best_iou': float(np.mean(ious_a))},
    {'model': DET_MODEL_B, 'mean_best_iou': float(np.mean(ious_b))},
]).sort_values('mean_best_iou', ascending=False)

metric_table

In [ ]:
# Find an image where the models differ meaningfully (example difference)
# Heuristic: pick image with largest absolute IoU gap
idx = int(np.argmax(np.abs(np.array(ious_a) - np.array(ious_b))))
print('Selected difference example index:', idx)
print('DETR best-IoU:', ious_a[idx])
print('YOLOS best-IoU:', ious_b[idx])

_ = draw_detections(images[idx], preds_a[idx], score_thr=0.4, title=f"DETR ({DET_MODEL_A}) - difference example")
_ = draw_detections(images[idx], preds_b[idx], score_thr=0.4, title=f"YOLOS ({DET_MODEL_B}) - difference example")

### Object detection model comparison discussion

**Models compared:**
- `facebook/detr-resnet-50` (DETR)
- `hustvl/yolos-tiny` (YOLOS Tiny)

**Why these models?**
- DETR is a widely-used baseline Transformer detector.
- YOLOS is a Transformer-based detector designed to be smaller/faster (tiny variant).

**Tradeoffs you should observe:**
- **Latency**: YOLOS Tiny is often faster/lighter.
- **Quality**: DETR can be more stable in some scenes; YOLOS may miss small objects depending on settings.

Model cards:
- https://huggingface.co/facebook/detr-resnet-50
- https://huggingface.co/hustvl/yolos-tiny

**Concrete difference example:** The notebook selects an image with the largest IoU gap and visualizes both predictions.

---
## Task 2: Image captioning (`image-to-text`)

We use a pretrained captioning model to generate captions for the same images.

### Visualization requirement
We will show at least 5 images with their generated captions.

In [ ]:
CAPTION_MODEL = 'nlpconnect/vit-gpt2-image-captioning'

# Preferred path: Hugging Face pipeline for image captioning.
# Some environments do not expose 'image-to-text'; if so, we fall back to direct model.generate.
USE_PIPELINE = True
try:
    captioner = pipeline('image-to-text', model=CAPTION_MODEL, device=DEVICE)
    print('Using caption pipeline task: image-to-text')
except Exception as e:
    USE_PIPELINE = False
    print('image-to-text pipeline unavailable; using direct generation fallback.')
    print('Reason:', e)

    from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer

    caption_model = VisionEncoderDecoderModel.from_pretrained(CAPTION_MODEL)
    caption_processor = ViTImageProcessor.from_pretrained(CAPTION_MODEL)
    caption_tokenizer = AutoTokenizer.from_pretrained(CAPTION_MODEL)

    caption_device = 'cuda' if DEVICE == 0 else 'cpu'
    caption_model.to(caption_device)


def _normalize_caption_item(item):
    """Return {'generated_text': ...} regardless of pipeline output shape."""
    if isinstance(item, list) and len(item) > 0:
        item = item[0]
    if isinstance(item, dict):
        txt = item.get('generated_text') or item.get('text') or str(item)
        return {'generated_text': txt}
    return {'generated_text': str(item)}


def caption_one(img):
    if USE_PIPELINE:
        out = captioner(img)
        return _normalize_caption_item(out)

    # Direct generation fallback for compatibility
    pixel_values = caption_processor(images=img, return_tensors='pt').pixel_values.to(caption_device)
    with torch.no_grad():
        output_ids = caption_model.generate(pixel_values, max_length=32, num_beams=4)
    txt = caption_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return {'generated_text': txt}


def caption_predict(imgs, batch_size=4):
    if USE_PIPELINE:
        try:
            out = captioner(imgs, batch_size=batch_size)
            if isinstance(out, dict):
                out = [out]
            if isinstance(out, list) and len(out) == len(imgs):
                return [_normalize_caption_item(x) for x in out]
        except Exception:
            pass

    # Safe fallback: one image at a time (works for both paths)
    return [caption_one(img) for img in imgs]


cap_preds = caption_predict(images, batch_size=4)
print('Caption outputs:', len(cap_preds))
print('Example output:', cap_preds[0] if len(cap_preds) else cap_preds)

In [ ]:
# Visualize captions (at least 5)
VIS_CAP = 5
n_show = min(VIS_CAP, len(images), len(cap_preds))
print('Showing captions for:', n_show, 'images')

for i in range(n_show):
    img = images[i].convert('RGB')

    item = cap_preds[i]
    # Depending on pipeline/task version, this can be a dict or a list of dicts.
    if isinstance(item, list) and len(item) > 0:
        item = item[0]

    if isinstance(item, dict):
        caption = item.get('generated_text') or item.get('text') or str(item)
    else:
        caption = str(item)

    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Caption: {caption}")
    plt.show()

In [ ]:
# Captioning latency (avg ms per image)
cap_lat = avg_latency(lambda imgs, batch_size: caption_predict(imgs, batch_size=batch_size), images, batch_size=4, repeats=1)
print(f"Captioning avg latency: {cap_lat*1000.0:.2f} ms/image")

## Checkpoint 2 summary

- **Two pipelines implemented**: object detection + image-to-text.
- **20 images processed** from a Hugging Face dataset.
- **Visualizations**:
  - Detection: bounding boxes + labels on ≥ 5 images.
  - Captioning: images + generated captions on ≥ 5 images.
- **Model comparison (object detection)**:
  - Latency table + a lightweight IoU-based quality proxy + a concrete example where models differ.